## Mart_Table (mart_delivery_performance) GOLD LAYER INSERTION

In [0]:
%sql
USE CATALOG olist_ecommerce_project;

### Importing Libraries

In [0]:
from pyspark.sql.functions import (
    sum as spark_sum, avg, count, countDistinct,
    round as spark_round, col, when, datediff
)

What this does:

Filters to only delivered orders (nulls excluded)
Counts late vs on-time
Calculates on-time delivery rate as a percentage (inverse of is_late)
Shows average delay in days and freight metrics

In [0]:
# Load tables
df_fact = spark.table("olist_ecommerce_project.gold.fact_orders")
df_date = spark.table("olist_ecommerce_project.gold.dim_date")
df_order_items = spark.table("olist_ecommerce_project.silver.slv_order_items")
df_sellers = spark.table("olist_ecommerce_project.gold.dim_sellers")

# Join: fact → date → order_items → sellers
df_delivery = (
    df_fact
    .join(df_date, on="date_key", how="inner")
    .join(df_order_items, on="order_id", how="inner")
    .join(df_sellers, on="seller_id", how="left")
)

# Filter to delivered orders only
df_delivered = df_delivery.filter(col("order_delivered_customer_date").isNotNull())

# Compute days_to_deliver inline
df_delivered = df_delivered.withColumn(
    "days_to_deliver",
    datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
)

# Aggregate by month
df_mart_delivery = (
    df_delivered
    .groupBy("year", "month_number", "month_name")
    .agg(
        count(col("order_id")).alias("total_orders_delivered"),
        spark_sum(when(col("is_late") == True, 1).otherwise(0)).alias("total_orders_late"),
        spark_round(
            spark_sum(when(col("is_late") == False, 1).otherwise(0)) / count(col("order_id")) * 100,
            2
        ).alias("on_time_delivery_rate_pct"),
        spark_round(avg("delivery_delay_days"), 2).alias("avg_delivery_delay_days"),
        spark_round(avg("days_to_deliver"), 2).alias("avg_days_to_deliver"),
        spark_round(avg("total_freight_value"), 2).alias("avg_freight_value"),
        spark_round(
            spark_sum("total_freight_value") / spark_sum("total_order_value") * 100,
            2
        ).alias("freight_to_revenue_ratio_pct")
    )
    .orderBy("year", "month_number")
)

print("mart_delivery_performance rows:", df_mart_delivery.count())
df_mart_delivery.show(10, truncate=False)


In [0]:
# Write to Gold
(
    df_mart_delivery.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.gold.mart_delivery_performance")
)

print("mart_delivery_performance written successfully")